# Módulo 3 · Práctica 3 — Agrupando Pokémon
**Diplomado en Ciencia de Datos · Generación 40**

**Objetivo:** formar grupos de Pokémon a partir de sus características y, con esos grupos, contestar tres preguntas:

1. ¿Cuál es el mejor grupo en general?
2. ¿Cuál es el peor grupo en general?
3. ¿Se puede armar un equipo de ensueño de 6 Pokémon que haga mucho daño y sea relativamente impermeable a cualquier otro equipo de 6?

**Plan de trabajo:**
1. Cargar el dataset.
2. Explorarlo rápido (nulos, distribuciones, legendarios).
3. Entender las dos ideas que sostienen todo: distancia + escalado y K-Means (con ejemplos mini hechos a mano).
4. Preparar las variables (limpieza, logaritmos, escalado) y decidir cuáles usamos para agrupar.
5. Decidir cuántos grupos (k).
6. Ajustar el modelo final y revisar qué tan estable es.
7. Perfilar y **nombrar** los grupos.
8. Visualizar los grupos (PCA y UMAP).
9. Describir cada grupo a detalle (estadísticas, tipos, debilidades).
10. Responder cuál es el mejor y el peor grupo (con un modelo de duelos).
11. Buscar el equipo de ensueño y ponerlo a prueba.
12. Conclusiones y limitaciones.

## 0. Instalación e importación de librerías

In [2]:
%pip install plotly nbformat scikit-learn umap-learn scipy pandas numpy -q


[notice] A new release of pip available: 22.3 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import warnings

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, silhouette_samples, davies_bouldin_score, adjusted_rand_score,
)

import umap

warnings.filterwarnings("ignore")   # umap/numba avisan cosas que no nos afectan

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PLANTILLA = "plotly_white"   # estilo de todas las gráficas
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. Carga del dataset


In [4]:
RUTA_ARCHIVO = "pokemon.csv"

if not os.path.isfile(RUTA_ARCHIVO):
    raise FileNotFoundError(f"No encuentro {RUTA_ARCHIVO}. Revisa RUTA_ARCHIVO.")

pokemon = pd.read_csv(RUTA_ARCHIVO)
print("Archivo encontrado:", RUTA_ARCHIVO)
print("pokemon:", pokemon.shape, "-> (Pokémon, columnas)")

Archivo encontrado: pokemon.csv
pokemon: (801, 41) -> (Pokémon, columnas)


In [5]:
# Show rápido de la tabla que leímos
display(pokemon.head())
print(pokemon.dtypes.value_counts())

,abilities,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,against_grass,against_ground,against_ice,against_normal,against_poison,against_psychic,against_rock,against_steel,against_water,attack,base_egg_steps,base_happiness,base_total,capture_rate,classfication,defense,experience_growth,height_m,hp,japanese_name,name,percentage_male,pokedex_number,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation,is_legendary
0,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,49,5120,70,318,45,Seed Pokémon,49,1059860,0.7,45,Fushigidaneフシギダネ,Bulbasaur,88.1,1,65,65,45,grass,poison,6.9,1,0
1,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,62,5120,70,405,45,Seed Pokémon,63,1059860,1.0,60,Fushigisouフシギソウ,Ivysaur,88.1,2,80,80,60,grass,poison,13.0,1,0
2,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,0.25,1.0,2.0,1.0,1.0,2.0,1.0,1.0,0.5,100,5120,70,625,45,Seed Pokémon,123,1059860,2.0,80,Fushigibanaフシギバナ,Venusaur,88.1,3,122,120,80,grass,poison,100.0,1,0
3,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,52,5120,70,309,45,Lizard Pokémon,43,1059860,0.6,39,Hitokageヒトカゲ,Charmander,88.1,4,60,50,65,fire,NaN,8.5,1,0
4,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,0.50,2.0,0.5,1.0,1.0,1.0,2.0,0.5,2.0,64,5120,70,405,45,Flame Pokémon,58,1059860,1.1,58,Lizardoリザード,Charmeleon,88.1,5,80,65,80,fire,NaN,19.0,1,0


float64    21
int64      13
str         7
Name: count, dtype: int64


---
## 2. Exploración rápida

Antes de agrupar nada, hay que saber qué tan limpios están los datos y cómo se distribuyen. Nos interesa:

1. ¿Qué columnas tienen nulos y qué significan?
2. ¿Cómo se ven las estadísticas de combate (y qué tanto cambian los legendarios)?
3. ¿Qué variables están muy sesgadas? Esto importa porque K-Means usa distancias.

In [6]:
# 2.1 Nulos, duplicados y cosas raras
nulos = pokemon.isnull().sum()
print("Columnas con nulos:")
display(nulos[nulos > 0].to_frame("n_nulos").assign(pct=lambda d: (d["n_nulos"] / len(pokemon) * 100).round(1)))

print("\nNúmeros de Pokédex duplicados:", pokemon["pokedex_number"].duplicated().sum())
print("Nombres duplicados           :", pokemon["name"].duplicated().sum())

# capture_rate debería ser numérica, pero viene como texto: buscamos quién no es un número puro
raros = pokemon[~pokemon["capture_rate"].astype(str).str.fullmatch(r"\d+")]
print("\nValores de capture_rate que no son un número:")
display(raros[["name", "capture_rate"]])

Columnas con nulos:


,n_nulos,pct
height_m,20,2.5
percentage_male,98,12.2
type2,384,47.9
weight_kg,20,2.5



Números de Pokédex duplicados: 0
Nombres duplicados           : 0

Valores de capture_rate que no son un número:


,name,capture_rate
773,Minior,30 (Meteorite)255 (Core)


**Qué significan los nulos (no todos son errores):**

- `type2` (384): son los Pokémon de un solo tipo. No falta nada.
- `percentage_male` (98): son los Pokémon **sin género**, como dice el diccionario. Tampoco falta nada.
- `height_m` y `weight_kg` (20 cada una): sí son datos faltantes (casi todos son formas de Alola). Los vamos a llenar con la mediana en la sección 4.
- `capture_rate` viene como texto porque **Minior** trae dos valores pegados (`30 (Meteorite)255 (Core)`). Nos quedamos con el primero.

In [7]:
# 2.2 Distribución de las estadísticas base
STATS = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed"]

largo = pokemon.melt(id_vars=["name", "is_legendary"], value_vars=STATS, var_name="estadistica", value_name="valor")
fig = px.box(
    largo, x="estadistica", y="valor", color="is_legendary", points="outliers",
    title="Estadísticas base (legendarios vs no legendarios)", template=PLANTILLA,
    labels={"is_legendary": "¿legendario?"},
)
fig.show()

print(f"Legendarios: {pokemon['is_legendary'].sum()} de {len(pokemon)} ({pokemon['is_legendary'].mean() * 100:.1f}%)")
print("base_total promedio -> no legendarios:", round(pokemon.loc[pokemon['is_legendary'] == 0, 'base_total'].mean(), 1),
      "| legendarios:", round(pokemon.loc[pokemon['is_legendary'] == 1, 'base_total'].mean(), 1))

Legendarios: 70 de 801 (8.7%)
base_total promedio -> no legendarios: 410.5 | legendarios: 615.3


In [8]:
# 2.3 Altura, peso y pasos de huevo: ¿están "bien portados"?
fig = make_subplots(rows=1, cols=3, subplot_titles=["height_m", "weight_kg", "base_egg_steps"])
for j, col in enumerate(["height_m", "weight_kg", "base_egg_steps"], start=1):
    fig.add_trace(go.Histogram(x=pokemon[col], nbinsx=50, showlegend=False), row=1, col=j)
fig.update_layout(title_text="Variables físicas y de huevo (escala original)", template=PLANTILLA, height=350)
fig.show()

print("Asimetría (skew) de cada variable; cerca de 0 = simétrica:")
print(pokemon[["height_m", "weight_kg", "base_egg_steps"]].skew().round(2).to_string())

Asimetría (skew) de cada variable; cerca de 0 = simétrica:
height_m          5.08
weight_kg         4.87
base_egg_steps    2.96


In [9]:
# 2.4 Tipos y generaciones
tipos = pd.concat([pokemon["type1"], pokemon["type2"].dropna()]).value_counts().reset_index()
tipos.columns = ["tipo", "n_pokemon"]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Pokémon por tipo (tipo1 + tipo2)", "Pokémon por generación"],
                    column_widths=[0.6, 0.4])
fig.add_trace(go.Bar(x=tipos["n_pokemon"], y=tipos["tipo"], orientation="h", showlegend=False), row=1, col=1)
por_gen = pokemon.groupby("generation").agg(total=("name", "size"), legendarios=("is_legendary", "sum")).reset_index()
fig.add_trace(go.Bar(x=por_gen["generation"], y=por_gen["total"], name="total"), row=1, col=2)
fig.add_trace(go.Bar(x=por_gen["generation"], y=por_gen["legendarios"], name="legendarios"), row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_layout(template=PLANTILLA, height=420, barmode="group")
fig.show()

**Cómo leer estas gráficas:**

- **Estadísticas:** los legendarios (70 de 801, el 8.7%) están muy por arriba del resto en las seis estadísticas: su `base_total` promedio es 615 contra 410 de los no legendarios. Más adelante vamos a ver si los grupos "descubren" esta diferencia por sí solos, ya que `is_legendary` **no** se usa para agrupar.
- **Altura, peso y pasos de huevo:** están muy sesgadas a la derecha (asimetría de 5.1, 4.9 y 3.0). La mayoría de los Pokémon son chicos y ligeros y unos pocos son enormes (Wailord mide 14.5 m y Groudon pesa 950 kg). Con distancias, esos pocos gigantes dominarían todo, así que vamos a usar logaritmo.
- **Tipos y generaciones:** *water* es el tipo más común (131 Pokémon contando tipo1 y tipo2) y *ice* de los menos (38). La generación 7 es la que más legendarios tiene (17 de 80).

---
## 3. Las dos ideas: distancia con escalado y K-Means

Para decir "estos dos Pokémon se parecen" necesitamos convertir cada uno en un **vector** (una lista de números) y medir qué tan cerca están. Usamos la **distancia euclidiana**:

$$d(A, B) = \sqrt{\sum_i (A_i - B_i)^2}$$

**Problema:** la distancia suma diferencias en las unidades originales. Una diferencia de 100 kg de peso pesa lo mismo que una diferencia de 100 puntos de ataque, aunque el ataque va de 5 a 185 y el peso de 0.1 a 999.9. Por eso **hay que escalar** (restar el promedio y dividir entre la desviación, o *z-score*) antes de agrupar.

Lo vemos con 3 Pokémon y 2 variables, primero **a mano** y luego con `sklearn` para confirmar:

In [10]:
# Ejemplo de juguete: 3 Pokémon, 2 variables -> [attack, weight_kg]
nombres_toy = ["Pikachu", "Jolteon", "Snorlax"]
toy = pokemon.set_index("name").loc[nombres_toy, ["attack", "weight_kg"]]
display(toy)


def distancia_manual(a, b):
    """Distancia euclidiana paso a paso: raíz de la suma de diferencias al cuadrado."""
    return np.sqrt(np.sum((a - b) ** 2))


# 1) Sin escalar: el peso (kg) manda por completo
v = toy.values
print("SIN ESCALAR")
print(f"  d(Pikachu, Jolteon)  = {distancia_manual(v[0], v[1]):6.1f}")
print(f"  d(Pikachu, Snorlax) = {distancia_manual(v[0], v[2]):6.1f}")

# 2) Escalando con la media y desviación de TODO el dataset (z-score)
escala = StandardScaler().fit(pokemon[["attack", "weight_kg"]])
z = escala.transform(toy)
print("\nESCALADO (z-score)")
print(f"  d(Pikachu, Jolteon)  = {distancia_manual(z[0], z[1]):6.2f}")
print(f"  d(Pikachu, Snorlax) = {distancia_manual(z[0], z[2]):6.2f}")

# Verificación con sklearn
from sklearn.metrics.pairwise import euclidean_distances
print("\nCon sklearn (escalado):", euclidean_distances(z)[0, 1:].round(2))

,attack,weight_kg
name,,
Pikachu,55,6.0
Jolteon,65,24.5
Snorlax,110,460.0


SIN ESCALAR
  d(Pikachu, Jolteon)  =   21.0
  d(Pikachu, Snorlax) =  457.3

ESCALADO (z-score)
  d(Pikachu, Jolteon)  =   0.35
  d(Pikachu, Snorlax) =   4.49

Con sklearn (escalado): [0.35 4.49]


**En palabras simples:** sin escalar, la distancia entre Pikachu y Snorlax es 457 y casi todo viene del peso (460 kg contra 6 kg); el ataque (110 contra 55) apenas se nota. Ya escalado, el peso y el ataque aportan en la misma medida (0.35 y 4.49 son distancias en "desviaciones estándar"). El orden se mantiene (Jolteon sigue más cerca de Pikachu que Snorlax), pero ahora los dos ejes cuentan.

Ahora la segunda idea, **K-Means**: dado un número de grupos *k*, busca *k* centros de forma que cada Pokémon quede en el grupo de su centro más cercano. El algoritmo repite dos pasos:

1. **Asignar:** cada punto se va al centro más cercano.
2. **Actualizar:** cada centro se mueve al promedio de los puntos que le tocaron.

Se repite hasta que nadie cambia de grupo. Lo vemos con 6 puntos y *k* = 2, una iteración a mano:

In [11]:
# K-Means a mano: 6 puntos en 2D, k = 2, una sola iteración
puntos = np.array([[1.0, 1.0], [1.5, 2.0], [1.0, 0.5],      # esquina de abajo-izquierda
                   [8.0, 8.0], [9.0, 9.5], [8.5, 9.0]])      # esquina de arriba-derecha
centros = np.array([[1.0, 1.0], [8.0, 8.0]])                 # arrancamos con 2 puntos como centros

# Paso 1: asignar cada punto al centro más cercano
distancias = np.array([[distancia_manual(p, c) for c in centros] for p in puntos])
asignacion = distancias.argmin(axis=1)
print("Asignación:", asignacion)

# Paso 2: mover cada centro al promedio de sus puntos
nuevos_centros = np.array([puntos[asignacion == j].mean(axis=0) for j in range(2)])
print("Nuevos centros:\n", nuevos_centros.round(2))

# Verificación con sklearn
km_toy = KMeans(n_clusters=2, init=centros, n_init=1).fit(puntos)
print("\nCon sklearn:\n", km_toy.cluster_centers_.round(2))

Asignación: [0 0 0 1 1 1]
Nuevos centros:
 [[1.17 1.17]
 [8.5  8.83]]

Con sklearn:
 [[1.17 1.17]
 [8.5  8.83]]


**En palabras simples:** los tres puntos de abajo-izquierda se van al primer centro, los tres de arriba-derecha al segundo, y cada centro se mueve al promedio de los suyos. Con Pokémon es exactamente lo mismo, pero con 11 dimensiones en lugar de 2. Dos cosas a tener presentes para el resto de la práctica:

- K-Means siempre te da *k* grupos aunque los datos no tengan grupos "naturales", por eso después revisamos qué tan buenos son (silhouette, estabilidad).
- Los grupos que forma son de forma más o menos redonda (cada uno gira alrededor de su centro).

---
## 4. Preparar las variables

### 4.1 y 4.2 Limpieza y logaritmos

Llenamos los nulos de `height_m` y `weight_kg` con la mediana, arreglamos `capture_rate` (Minior) y le aplicamos logaritmo a las tres variables muy sesgadas (altura, peso y pasos de huevo).

In [12]:
df = pokemon.copy()

# 4.1 Limpieza
# capture_rate: Minior trae "30 (Meteorite)255 (Core)"; nos quedamos con el primer número (30)
df["capture_rate"] = pd.to_numeric(df["capture_rate"].astype(str).str.extract(r"^(\d+)")[0])

# height_m y weight_kg: 20 nulos (formas de Alola). Los llenamos con la mediana
for col in ["height_m", "weight_kg"]:
    df[col] = df[col].fillna(df[col].median())

# 4.2 Transformación logarítmica de las variables muy sesgadas
df["log_height"] = np.log(df["height_m"])
df["log_weight"] = np.log(df["weight_kg"])
df["log_egg_steps"] = np.log(df["base_egg_steps"])

print("Asimetría antes y después del logaritmo:")
print(pd.DataFrame({
    "antes":   [df["height_m"].skew(), df["weight_kg"].skew(), df["base_egg_steps"].skew()],
    "despues": [df["log_height"].skew(), df["log_weight"].skew(), df["log_egg_steps"].skew()],
}, index=["altura", "peso", "pasos_huevo"]).round(2))

Asimetría antes y después del logaritmo:
             antes  despues
altura        5.15    -0.01
peso          4.94    -0.56
pasos_huevo   2.96     1.87


**Ojo:** el logaritmo baja mucho la asimetría de altura (5.15 → -0.01) y peso (4.94 → -0.56). Los pasos de huevo mejoran menos (2.96 → 1.87) porque la variable es casi dos "mundos": los Pokémon normales (~5,000 pasos) y los legendarios (~25,000). Lo dejamos así porque esa separación es información real.

### 4.3 ¿Qué variables usamos para agrupar?

Usamos **11 variables**: las 6 estadísticas de combate, altura y peso (en log), `capture_rate`, pasos de huevo (en log) y `experience_growth`. Y dejamos fuera, a propósito:

- **`base_total`:** es la suma de las 6 estadísticas, no aporta nada nuevo y les daría más peso del que deben tener.
- **Las 18 `against_*`:** describen el *tipo* del Pokémon (son combinaciones de 18 números que dependen casi por completo de `type1` y `type2`). Meter 18 columnas de golpe haría que los grupos salgan por tipo y no por poder (lo comprobamos en 4.4). Las usamos después, para **perfilar** y para los duelos.
- **`is_legendary`, `generation`, `base_happiness`, `percentage_male`, tipos:** no son características de combate. Las dejamos para *describir* los grupos y ver si coinciden con lo que el modelo encontró sin que se lo dijéramos.

Y escalamos todo con z-score (sección 3):

In [13]:
# 4.3 Variables que usamos para agrupar
VARS_STATS = STATS                                           # 6 estadísticas de combate
VARS_FISICAS = ["log_height", "log_weight"]                  # tamaño
VARS_OTRAS = ["capture_rate", "log_egg_steps", "experience_growth"]   # qué tan "especial" es
VARS_CLUSTER = VARS_STATS + VARS_FISICAS + VARS_OTRAS

scaler = StandardScaler()
X = scaler.fit_transform(df[VARS_CLUSTER])
print("Matriz para agrupar:", X.shape, "-> (Pokémon, variables)")
print("Promedios ~0:", X.mean(axis=0).round(2).max(), "| desviaciones ~1:", X.std(axis=0).round(2).min())

Matriz para agrupar: (801, 11) -> (Pokémon, variables)
Promedios ~0: 0.0 | desviaciones ~1: 1.0


In [14]:
corr = df[VARS_CLUSTER].corr().round(2)
fig = px.imshow(corr, text_auto=True, zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
                title="Correlación entre las variables usadas", template=PLANTILLA, height=620)
fig.show()

**Cómo leer la matriz:**

- Las 6 estadísticas se correlacionan positivamente entre sí pero de forma moderada (entre 0.01 y 0.53): un Pokémon fuerte en una no necesariamente lo es en las demás. Esto es lo que hace posible que existan *estilos* de combate.
- `capture_rate` tiene correlación **negativa** con todas las estadísticas (entre -0.4 y -0.5): entre más fuerte, más difícil de capturar.
- La altura y el peso (en log) van muy juntos (0.81) y a su vez se relacionan con HP y ataque (alrededor de 0.5). Es la correlación más alta de la matriz: tamaño y peso casi cuentan la misma historia, y por eso entre las dos pesan un poco más que una sola variable.
- Los pasos de huevo y `experience_growth` también van juntos (0.38), por los legendarios.

In [15]:
# 4.4 ¿Y por qué no usamos TODAS las columnas (incluyendo las 18 against_*)?
# Lo comprobamos con el silhouette (qué tan bien separados quedan los grupos) para k = 6
AGAINST = [c for c in df.columns if c.startswith("against_")]

conjuntos = {
    "Solo stats (6)":                       VARS_STATS,
    "Stats + físicas (8)":                  VARS_STATS + VARS_FISICAS,
    "Stats + físicas + otras (11) <- usado": VARS_CLUSTER,
    "Solo against_* (18)":                  AGAINST,
    "Todo junto (29)":                      VARS_CLUSTER + AGAINST,
}

filas = []
for nombre, columnas in conjuntos.items():
    Xc = StandardScaler().fit_transform(df[columnas])
    etiquetas = KMeans(n_clusters=6, n_init=20, random_state=RANDOM_STATE).fit_predict(Xc)
    filas.append({"conjunto": nombre, "n_variables": len(columnas), "silhouette_k6": silhouette_score(Xc, etiquetas)})

comparacion = pd.DataFrame(filas).round(3)
display(comparacion)

,conjunto,n_variables,silhouette_k6
0,Solo stats (6),6,0.225
1,Stats + físicas (8),8,0.145
2,Stats + físicas + otras (11) <- usado,11,0.148
3,Solo against_* (18),18,0.227
4,Todo junto (29),29,0.118


**Cómo leer la tabla (silhouette, de -1 a 1; mientras más alto, grupos más separados):**

- Con solo las 6 estadísticas (0.225) los grupos quedan más separados que con las 11 variables (0.148). Es lógico: entre más dimensiones, más "ruido" para separar.
- Las 18 `against_*` solas (0.227) también separan bien, porque los tipos forman bloques muy marcados, pero eso agruparía por tipo (todos los *fire* juntos, etc.) y no por poder. Y al mezclarlas con todo (0.118) el resultado empeora.

**Decisión:** usamos las 11 variables aunque el silhouette sea menor que con solo estadísticas, porque queremos que el tamaño y la "rareza" (captura, huevo, experiencia) también cuenten para describir a un Pokémon. Es una decisión de qué queremos que signifique "grupo", no una métrica que ganó.

---
## 5. ¿Cuántos grupos?

No hay una respuesta única, así que miramos tres cosas para *k* de 2 a 10:

- **Inercia** (suma de distancias al cuadrado al centro): siempre baja al subir *k*; buscamos un "codo".
- **Silhouette:** qué tan bien separados están los grupos (más alto = mejor).
- **Davies-Bouldin:** qué tan parecidos son entre sí los grupos (más bajo = mejor).

In [16]:
# 5.1 Probamos k = 2, ..., 10
filas = []
for k in range(2, 11):
    km_k = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit(X)
    filas.append({
        "k": k,
        "inercia": km_k.inertia_,
        "silhouette": silhouette_score(X, km_k.labels_),
        "davies_bouldin": davies_bouldin_score(X, km_k.labels_),
    })
metricas_k = pd.DataFrame(filas)

fig = make_subplots(rows=1, cols=3, subplot_titles=["Inercia (codo)", "Silhouette (más alto = mejor)", "Davies-Bouldin (más bajo = mejor)"])
for j, col in enumerate(["inercia", "silhouette", "davies_bouldin"], start=1):
    fig.add_trace(go.Scatter(x=metricas_k["k"], y=metricas_k[col], mode="lines+markers", showlegend=False), row=1, col=j)
fig.update_xaxes(title_text="k", dtick=1)
fig.update_layout(template=PLANTILLA, height=380, title_text="¿Cuántos grupos?")
fig.show()

display(metricas_k.round(3))

,k,inercia,silhouette,davies_bouldin
0,2,6277.032,0.253,1.427
1,3,5402.788,0.248,1.450
2,4,5000.413,0.191,1.811
3,5,4714.415,0.150,1.880
4,6,4465.185,0.145,1.880
5,7,4266.324,0.146,1.796
6,8,4113.095,0.146,1.733
7,9,3965.524,0.156,1.660
8,10,3810.553,0.154,1.590


In [17]:
# 5.2 Para k = 2, 3, 4, 6 y 7: tamaño de los grupos y base_total promedio de cada uno
for k in [2, 3, 4, 6, 7]:
    km_k = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE).fit(X)
    por_grupo = df.groupby(km_k.labels_)["base_total"].agg(["size", "mean"]).sort_values("mean")
    print(f"k = {k}: silhouette = {silhouette_score(X, km_k.labels_):.3f}")
    print("   tamaños     :", por_grupo["size"].tolist())
    print("   base_total  :", por_grupo["mean"].round(0).astype(int).tolist(), "(de menor a mayor)")

k = 2: silhouette = 0.253
   tamaños     : [353, 448]
   base_total  : [321, 513] (de menor a mayor)
k = 3: silhouette = 0.248
   tamaños     : [311, 409, 81]
   base_total  : [309, 482, 616] (de menor a mayor)
k = 4: silhouette = 0.191
   tamaños     : [297, 245, 182, 77]
   base_total  : [304, 480, 481, 618] (de menor a mayor)
k = 6: silhouette = 0.145
   tamaños     : [198, 161, 109, 129, 133, 71]
   base_total  : [289, 363, 481, 492, 503, 627] (de menor a mayor)
k = 7: silhouette = 0.146
   tamaños     : [176, 167, 108, 127, 131, 28, 64]
   base_total  : [285, 350, 480, 494, 501, 502, 630] (de menor a mayor)


**Cómo leer estas gráficas:**

- **Inercia:** baja de forma suave, **no hay un codo claro**.
- **Silhouette y Davies-Bouldin:** los mejores valores son con *k* = 2 y 3 (silhouette de 0.25); baja a 0.19 con *k* = 4 y desde *k* = 5 se queda **plano** (alrededor de 0.15) hasta *k* = 10.

**Pero ¿qué son los grupos con *k* = 2 o 3?** Con *k* = 2 solo separa a los débiles (353 Pokémon, `base_total` ≈ 321) de los fuertes (448, ≈ 513); con *k* = 3 aparece además una élite de 81 Pokémon (≈ 616) donde 62 son legendarios. Es decir, son solo **niveles de poder**: no nos dice nada de *cómo* pelean.

**Decisión: *k* = 6.** Como el silhouette ya no mejora ni empeora entre 5 y 10, escogemos por interpretabilidad: con 6 la "zona media" se parte en estilos de combate distintos (rápidos, defensivos, físicos), que es justo lo que se pide perfilar. Con *k* = 7 sale un grupo de apenas 28 Pokémon que no aporta un perfil nuevo.

**Ojo:** *k* = 6 no es "el óptimo matemático", es una decisión razonable. Por eso en la sección 6 revisamos qué tan estable es.

---
## 6. Modelo final: K-Means con *k* = 6

Una advertencia práctica: K-Means numera los grupos al azar (el "grupo 3" de una corrida puede ser el "5" en otra). Para no depender de eso, los reordenamos por `base_total` promedio: **0 = el más débil, 5 = el más fuerte**.

In [18]:
# 6.1 Modelo final
K_FINAL = 6
kmeans = KMeans(n_clusters=K_FINAL, n_init=50, random_state=RANDOM_STATE).fit(X)
df["grupo_id"] = kmeans.labels_

# Las etiquetas de K-Means son arbitrarias (el "grupo 3" de hoy puede ser el "5" mañana),
# así que ordenamos los grupos de menor a mayor base_total y renombramos 0..5
orden = df.groupby("grupo_id")["base_total"].mean().sort_values().index
reetiqueta = {viejo: nuevo for nuevo, viejo in enumerate(orden)}
df["grupo_id"] = df["grupo_id"].map(reetiqueta)

print("Tamaño de cada grupo (0 = más débil, 5 = más fuerte):")
print(df["grupo_id"].value_counts().sort_index().to_string())
print(f"\nSilhouette global: {silhouette_score(X, df['grupo_id']):.3f}")

Tamaño de cada grupo (0 = más débil, 5 = más fuerte):
grupo_id
0    198
1    161
2    109
3    129
4    133
5     71

Silhouette global: 0.145


In [19]:
# 6.2 ¿Qué tan estable es esta solución?
# (a) Cambiando la semilla aleatoria
aris_semilla = []
for semilla in range(10):
    otro = KMeans(n_clusters=K_FINAL, n_init=50, random_state=semilla).fit_predict(X)
    aris_semilla.append(adjusted_rand_score(df["grupo_id"], otro))

# (b) Re-entrenando con el 80% de los Pokémon al azar
rng = np.random.default_rng(RANDOM_STATE)
aris_boot = []
for _ in range(30):
    idx = rng.choice(len(X), size=int(len(X) * 0.8), replace=False)
    otro = KMeans(n_clusters=K_FINAL, n_init=20, random_state=RANDOM_STATE).fit_predict(X[idx])
    aris_boot.append(adjusted_rand_score(df["grupo_id"].values[idx], otro))

# (c) ¿Otros algoritmos encuentran lo mismo?
ward = AgglomerativeClustering(n_clusters=K_FINAL, linkage="ward").fit_predict(X)
gmm = GaussianMixture(n_components=K_FINAL, n_init=5, random_state=RANDOM_STATE).fit_predict(X)

print(f"(a) ARI entre semillas  : promedio {np.mean(aris_semilla):.3f} (mínimo {np.min(aris_semilla):.3f})")
print(f"(b) ARI con 80% de datos: promedio {np.mean(aris_boot):.3f} (mínimo {np.min(aris_boot):.3f})")
print(f"(c) ARI K-Means vs Ward : {adjusted_rand_score(df['grupo_id'], ward):.3f} | vs mezcla gaussiana: {adjusted_rand_score(df['grupo_id'], gmm):.3f}")
print(f"    Silhouette -> K-Means {silhouette_score(X, df['grupo_id']):.3f} | Ward {silhouette_score(X, ward):.3f} | GMM {silhouette_score(X, gmm):.3f}")

(a) ARI entre semillas  : promedio 0.947 (mínimo 0.894)
(b) ARI con 80% de datos: promedio 0.821 (mínimo 0.631)
(c) ARI K-Means vs Ward : 0.523 | vs mezcla gaussiana: 0.248
    Silhouette -> K-Means 0.145 | Ward 0.117 | GMM 0.071


**Cómo leer estas pruebas (ARI = 1 significa que dos agrupaciones son idénticas; 0, que coinciden como si fueran al azar):**

- **(a) Cambiando la semilla:** ARI promedio de 0.95 (mínimo 0.89). El resultado casi no depende de la semilla.
- **(b) Con el 80% de los Pokémon:** ARI promedio de 0.82 (mínimo 0.63). Es razonablemente estable, aunque en algunas muestras cambia más.
- **(c) Otros algoritmos:** Ward coincide a medias (0.52) y la mezcla gaussiana poco (0.25).

**En palabras simples:** los 6 grupos son un resumen útil, pero **no son "islas" naturales**: los Pokémon forman más bien una nube continua (de débil a fuerte) y K-Means la corta en 6 pedazos. Por eso el silhouette global es bajo (0.145). Hay que tener esto presente al interpretar.

In [20]:
# 6.3 Silhouette de cada Pokémon, resumido por grupo
df["silhouette"] = silhouette_samples(X, df["grupo_id"])
resumen_sil = df.groupby("grupo_id")["silhouette"].agg(["mean", lambda s: (s < 0).mean()])
resumen_sil.columns = ["silhouette_promedio", "pct_mal_asignados"]
display(resumen_sil.round(3))

,silhouette_promedio,pct_mal_asignados
grupo_id,,
0,0.198,0.000
1,0.121,0.056
2,0.029,0.358
3,0.090,0.233
4,0.208,0.000
5,0.216,0.127


**Cómo leer esta tabla:** un silhouette negativo significa que el Pokémon está más cerca de otro grupo que del suyo.

- **Novatos (0.20), Relámpagos (0.21) y Titanes (0.22)** son los grupos mejor definidos.
- **Murallas (0.03)** es el grupo más difuso: el 36% de sus Pokémon queda más cerca de otro grupo. Le sigue **Gladiadores** (0.09, con 23% mal asignados). Es decir, Murallas y Gladiadores se mezclan bastante con sus vecinos.
- Los Titanes tienen un silhouette alto, pero 13% de ellos queda más cerca de otro grupo: son los de la frontera con el "medio alto" (Dragonite, Wailord, Regirock, Uxie, Volcarona...).

---
## 7. Perfilamiento y nombres de los grupos

Para ponerle nombre a cada grupo miramos primero sus promedios. Los nombres **no se asignan por el número del grupo** (que es arbitrario) sino por una regla sobre su perfil: el de mayor `base_total`, el de menor, el más veloz de los que quedan, el que más defiende, el de mayor ataque físico, y el que sobra.

In [21]:
# 7. Perfilamiento: primero una tabla con el promedio de cada grupo
COLS_PERFIL = STATS + ["height_m", "weight_kg", "capture_rate", "base_egg_steps", "experience_growth", "base_total"]
perfil = df.groupby("grupo_id")[COLS_PERFIL].mean()
perfil["n_pokemon"] = df.groupby("grupo_id").size()
perfil["pct_legendarios"] = df.groupby("grupo_id")["is_legendary"].mean() * 100
perfil["def_total"] = perfil["defense"] + perfil["sp_defense"]
display(perfil.round(1))


def nombrar_grupos(perfil):
    """Asigna un nombre a cada grupo según sus rasgos más marcantes (no según su número, que es arbitrario)."""
    nombres, restantes = {}, list(perfil.index)

    def toma(columna, nombre, mayor=True):
        elegido = perfil.loc[restantes, columna].idxmax() if mayor else perfil.loc[restantes, columna].idxmin()
        nombres[elegido] = nombre
        restantes.remove(elegido)

    toma("base_total", "Titanes", mayor=True)        # el más fuerte en general
    toma("base_total", "Novatos", mayor=False)       # el más débil en general
    toma("speed", "Relámpagos", mayor=True)          # de los que quedan, el más veloz
    toma("def_total", "Murallas", mayor=True)        # el que más aguanta
    toma("attack", "Gladiadores", mayor=True)        # el de más ataque físico
    nombres[restantes[0]] = "Reclutas"               # el que sobra: nivel medio-bajo
    return nombres


NOMBRES = nombrar_grupos(perfil)
df["grupo"] = df["grupo_id"].map(NOMBRES)
ORDEN = [NOMBRES[i] for i in sorted(NOMBRES)]        # de menor a mayor base_total
print("Nombres asignados:", NOMBRES)

,hp,attack,defense,sp_attack,sp_defense,speed,height_m,weight_kg,capture_rate,base_egg_steps,experience_growth,base_total,n_pokemon,pct_legendarios,def_total
grupo_id,,,,,,,,,,,,,,,
0,48.3,50.0,49.4,45.3,47.1,49.1,0.5,12.3,209.6,4667.5,999113.1,289.2,198,0.0,96.5
1,57.4,66.2,58.7,61.6,59.2,60.3,0.7,18.2,85.4,5883.2,1077203.6,363.5,161,1.2,118.0
2,77.6,70.4,96.4,85.8,101.5,49.2,1.2,68.1,67.9,5507.5,946582.4,480.9,109,1.8,197.8
3,88.8,107.7,96.9,64.5,73.0,61.3,1.7,119.1,59.2,5913.8,1125705.0,492.3,129,1.6,169.9
4,70.7,92.4,68.7,92.6,76.1,102.8,1.3,44.3,56.1,5283.6,1028831.1,503.3,133,3.8,144.8
5,100.2,112.0,100.2,115.7,103.4,95.2,2.6,203.5,18.2,25672.1,1247458.9,626.6,71,83.1,203.5


Nombres asignados: {np.int64(5): 'Titanes', np.int64(0): 'Novatos', np.int64(4): 'Relámpagos', np.int64(2): 'Murallas', np.int64(3): 'Gladiadores', 1: 'Reclutas'}


In [22]:
COLORES = dict(zip(ORDEN, px.colors.qualitative.Safe))

perfil.index = perfil.index.map(NOMBRES)   # ahora las tablas se leen con nombre
resumen = df.groupby("grupo").agg(n=("name", "size"), base_total=("base_total", "mean"),
                                  pct_legendarios=("is_legendary", lambda s: s.mean() * 100)).loc[ORDEN]
display(resumen.round(1))

,n,base_total,pct_legendarios
grupo,,,
Novatos,198,289.2,0.0
Reclutas,161,363.5,1.2
Murallas,109,480.9,1.8
Gladiadores,129,492.3,1.6
Relámpagos,133,503.3,3.8
Titanes,71,626.6,83.1


**Los 6 grupos y por qué se llaman así:**

| Grupo | Pokémon | `base_total` | Rasgo principal |
|---|:---:|:---:|---|
| **Novatos** | 198 | 289 | Los más débiles en todo, los más fáciles de capturar (`capture_rate` de 210) |
| **Reclutas** | 161 | 363 | Un escalón arriba de los Novatos; nivel medio-bajo en todo, sin una especialidad clara |
| **Murallas** | 109 | 481 | Defensa y defensa especial altas (198 sumadas), pero los más lentos (49) |
| **Gladiadores** | 129 | 492 | Golpean fuerte (ataque de 108), con mucho HP, grandes y pesados (119 kg) |
| **Relámpagos** | 133 | 503 | Los más veloces (103) con buen ataque especial, y poca defensa |
| **Titanes** | 71 | 627 | Los más altos en todas las estadísticas; 83% son legendarios |

Notamos que **no usamos `is_legendary` para agrupar** y aun así el grupo más fuerte salió casi puro de legendarios, un buen indicio de que las variables que elegimos capturan algo real.

---
## 8. Visualización de los grupos

Tenemos 11 dimensiones y solo podemos ver 2 o 3. Usamos dos herramientas:

- **PCA:** gira los ejes para quedarnos con las direcciones donde hay más variación. Es lineal y los ejes se pueden interpretar.
- **UMAP:** proyecta cuidando que los vecinos cercanos sigan cercanos. Los ejes no significan nada, pero suele separar mejor los grupos.

### 8.1 PCA

In [23]:
# 8.1 PCA: de 11 dimensiones a 2 y 3 para poder "verlos"
pca = PCA(n_components=3, random_state=RANDOM_STATE).fit(X)
P = pca.transform(X)
df["pc1"], df["pc2"], df["pc3"] = P[:, 0], P[:, 1], P[:, 2]
print("Varianza explicada por componente:", pca.explained_variance_ratio_.round(3), "| acumulada (2):", pca.explained_variance_ratio_[:2].sum().round(3))

# ¿Qué significa cada componente? (pesos de cada variable)
cargas = pd.DataFrame(pca.components_[:3].T, index=VARS_CLUSTER, columns=["PC1", "PC2", "PC3"]).round(2)
display(cargas)

Varianza explicada por componente: [0.426 0.112 0.096] | acumulada (2): 0.538


,PC1,PC2,PC3
hp,0.31,-0.09,0.15
attack,0.33,-0.04,0.12
defense,0.28,-0.41,-0.13
sp_attack,0.31,0.35,-0.28
sp_defense,0.31,-0.07,-0.42
speed,0.21,0.59,-0.21
log_height,0.37,-0.22,0.11
log_weight,0.34,-0.37,0.19
capture_rate,-0.36,-0.07,0.22
log_egg_steps,0.27,0.30,0.24


In [24]:
fig = px.scatter(
    df, x="pc1", y="pc2", color="grupo", category_orders={"grupo": ORDEN}, color_discrete_map=COLORES,
    hover_name="name", hover_data={"type1": True, "type2": True, "base_total": True, "pc1": False, "pc2": False},
    title="Los 801 Pokémon en el plano de los 2 primeros componentes (PCA)", template=PLANTILLA, height=620,
    labels={"pc1": f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% de la varianza)", "pc2": f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)"},
)
fig.update_traces(marker=dict(size=7, opacity=0.8))
fig.show()

In [25]:
fig = px.scatter_3d(
    df, x="pc1", y="pc2", z="pc3", color="grupo", category_orders={"grupo": ORDEN}, color_discrete_map=COLORES,
    hover_name="name", title="Vista 3D (PCA)", template=PLANTILLA, height=700,
)
fig.update_traces(marker=dict(size=3, opacity=0.8))
fig.show()

In [26]:
# 8.2 UMAP: otra forma de proyectar, que cuida más la vecindad local
reductor = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=RANDOM_STATE)
U = reductor.fit_transform(X)
df["u1"], df["u2"] = U[:, 0], U[:, 1]

fig = px.scatter(
    df, x="u1", y="u2", color="grupo", category_orders={"grupo": ORDEN}, color_discrete_map=COLORES,
    hover_name="name", hover_data={"type1": True, "type2": True, "base_total": True, "u1": False, "u2": False},
    title="Los 801 Pokémon proyectados con UMAP", template=PLANTILLA, height=620,
)
fig.update_traces(marker=dict(size=7, opacity=0.8))
fig.show()

In [27]:
# 8.3 ¿Qué tan bien "se ven" los grupos en cada proyección?
# Para cada Pokémon miramos a sus 10 vecinos más cercanos en el plano 2D y contamos cuántos son de su mismo grupo
from sklearn.neighbors import NearestNeighbors


def pureza_vecindad(coordenadas, etiquetas, vecinos=10):
    """Porcentaje promedio de vecinos (en la proyección) que pertenecen al mismo grupo."""
    nn = NearestNeighbors(n_neighbors=vecinos + 1).fit(coordenadas)
    _, indices = nn.kneighbors(coordenadas)
    vecinos_idx = indices[:, 1:]                       # quitamos al propio Pokémon
    et = np.asarray(etiquetas)
    return (et[vecinos_idx] == et[:, None]).mean()


print(f"PCA 2D : {pureza_vecindad(df[['pc1', 'pc2']].values, df['grupo_id']) * 100:.1f}%")
print(f"UMAP 2D: {pureza_vecindad(df[['u1', 'u2']].values, df['grupo_id']) * 100:.1f}%")
print(f"Espacio original (11 variables, sin proyectar): {pureza_vecindad(X, df['grupo_id']) * 100:.1f}%")
print(f"Referencia al azar (según tamaño de los grupos): {(df['grupo_id'].value_counts(normalize=True) ** 2).sum() * 100:.1f}%")

PCA 2D : 66.4%
UMAP 2D: 83.2%
Espacio original (11 variables, sin proyectar): 80.9%
Referencia al azar (según tamaño de los grupos): 18.1%


**Cómo leer las proyecciones:**

- **PC1 (42.6% de la varianza):** es el "tamaño y poder general". Todas las estadísticas, la altura y el peso pesan positivo, y `capture_rate` negativo. De izquierda a derecha van de Novatos a Titanes.
- **PC2 (11.2%):** separa a los **rápidos y especiales** (arriba: `speed` 0.59, `sp_attack` 0.35) de los **pesados y defensivos** (abajo: `defense` -0.41, `weight` -0.37). Aquí se reparten Relámpagos, Murallas y Gladiadores.
- **PC3 (9.6%):** es casi solo `experience_growth` (0.70).
- Con 2 componentes solo vemos el **54%** de la información y en el plano se encima mucho. Se ve en la pureza de vecindad (8.3): en PCA 2D el 66% de los vecinos de un Pokémon son de su mismo grupo, mientras que en UMAP es el 83% (al azar sería 18%). UMAP incluso conserva la vecindad igual de bien que el espacio original de 11 variables (81%).

Por eso para **mostrar** los grupos en la presentación conviene UMAP, y PCA para **explicar qué significan los ejes**.

---
## 9. Perfil detallado de cada grupo

### 9.1 Estadísticas

Un mapa de calor con el perfil de cada grupo en "desviaciones estándar" respecto al promedio de todos los Pokémon (0 = igual al promedio), y un radar con las 6 estadísticas de combate.

In [28]:
# 9.1 Perfil de cada grupo en "desviaciones estándar" respecto al promedio de todos los Pokémon
z = (df.groupby("grupo")[VARS_CLUSTER].mean().loc[ORDEN] - df[VARS_CLUSTER].mean()) / df[VARS_CLUSTER].std()

fig = px.imshow(z.T, text_auto=".1f", color_continuous_scale="RdBu_r", zmin=-2.5, zmax=2.5, aspect="auto",
                title="Perfil de los grupos (0 = promedio global; rojo = por encima, azul = por debajo)",
                template=PLANTILLA, height=620)
fig.show()

In [29]:
fig = go.Figure()
for g in ORDEN:
    valores = df.loc[df["grupo"] == g, STATS].mean()
    fig.add_trace(go.Scatterpolar(r=list(valores) + [valores.iloc[0]], theta=STATS + [STATS[0]], name=g,
                                  line=dict(color=COLORES[g]), fill="toself", opacity=0.35))
fig.update_layout(title="Estadísticas base promedio por grupo", template=PLANTILLA, height=560,
                  polar=dict(radialaxis=dict(visible=True, range=[0, 125])))
fig.show()

In [30]:
# 9.2 ¿Qué tipos caracterizan a cada grupo?
largo_tipos = pd.concat([
    df[["grupo", "type1"]].rename(columns={"type1": "tipo"}),
    df.loc[df["type2"].notnull(), ["grupo", "type2"]].rename(columns={"type2": "tipo"}),
], ignore_index=True)
conteo = pd.crosstab(largo_tipos["grupo"], largo_tipos["tipo"]).loc[ORDEN]
participacion_global = conteo.sum() / conteo.values.sum()
participacion_grupo = conteo.div(conteo.sum(axis=1), axis=0)
lift = participacion_grupo.div(participacion_global, axis=1)   # >1 = sobrerrepresentado

# Mostramos, por grupo, los 3 tipos con mayor lift (pidiendo al menos 5 Pokémon de ese tipo)
destacados = {}
for g in ORDEN:
    candidatos = lift.loc[g][conteo.loc[g] >= 5].sort_values(ascending=False).head(3)
    destacados[g] = ", ".join(f"{t} (x{v:.1f})" for t, v in candidatos.items())
display(pd.Series(destacados, name="tipos sobrerrepresentados").to_frame())

fig = px.imshow(lift.round(2), text_auto=".1f", color_continuous_scale="RdBu_r", zmin=0, zmax=2, aspect="auto",
                title="Lift de cada tipo por grupo (1 = igual que en todo el dataset)", template=PLANTILLA, height=420)
fig.show()

,tipos sobrerrepresentados
Novatos,"bug (x1.7), electric (x1.5), poison (x1.5)"
Reclutas,"fire (x1.7), grass (x1.5), dragon (x1.2)"
Murallas,"fairy (x1.9), steel (x1.7), water (x1.6)"
Gladiadores,"ground (x2.4), fighting (x2.3), rock (x2.2)"
Relámpagos,"fire (x1.7), flying (x1.7), dark (x1.5)"
Titanes,"dragon (x4.0), psychic (x2.3), steel (x2.0)"


In [31]:
# 9.3 Perfil defensivo: daño promedio que recibe cada grupo por tipo de ataque (1 = normal, >1 = débil)
against_prom = df.groupby("grupo")[AGAINST].mean().loc[ORDEN]
against_prom.columns = [c.replace("against_", "") for c in against_prom.columns]

fig = px.imshow(against_prom.round(2), text_auto=".2f", color_continuous_scale="RdBu_r", color_continuous_midpoint=1, aspect="auto",
                title="Daño recibido promedio por tipo de ataque", template=PLANTILLA, height=420)
fig.show()

In [32]:
# 9.4 Ejemplos de cada grupo y rasgos extra (los que NO se usaron para agrupar)
filas = []
for g in ORDEN:
    sub = df[df["grupo"] == g]
    filas.append({
        "grupo": g,
        "ejemplos (mayor base_total)": ", ".join(sub.nlargest(5, "base_total")["name"]),
        "% con 2 tipos": round(sub["type2"].notnull().mean() * 100),
        "% sin género": round(sub["percentage_male"].isnull().mean() * 100),
        "felicidad base": round(sub["base_happiness"].mean(), 1),
        "generación más común": int(sub["generation"].mode()[0]),
    })
display(pd.DataFrame(filas).set_index("grupo"))

# Los Titanes no son solo legendarios: ¿quiénes son los que no lo son?
print("Titanes NO legendarios:", ", ".join(df[(df["grupo"] == "Titanes") & (df["is_legendary"] == 0)]["name"]))
print("Legendarios que NO cayeron en Titanes:", ", ".join(df[(df["grupo"] != "Titanes") & (df["is_legendary"] == 1)]["name"]))

,ejemplos (mayor base_total),% con 2 tipos,% sin género,felicidad base,generación más común
grupo,,,,,
Novatos,"Togedemaru, Dedenne, Emolga, Watchog, Plusle",43,5,69.9,1
Reclutas,"Kartana, Floette, Rotom, Comfey, Phione",45,4,65.8,5
Murallas,"Blastoise, Venusaur, Wishiwashi, Ampharos, Slo...",59,10,69.7,3
Gladiadores,"Slaking, Gyarados, Swampert, Aggron, Gallade",56,4,66.7,5
Relámpagos,"Diancie, Greninja, Charizard, Sceptile, Blaziken",58,8,68.6,1
Titanes,"Mewtwo, Rayquaza, Kyogre, Groudon, Arceus",65,79,36.3,5


Titanes NO legendarios: Dragonite, Tyranitar, Wailord, Salamence, Metagross, Garchomp, Hydreigon, Volcarona, Goodra, Type: Null, Silvally, Kommo-o
Legendarios que NO cayeron en Titanes: Azelf, Heatran, Manaphy, Diancie, Tapu Koko, Tapu Lele, Tapu Bulu, Tapu Fini, Cosmog, Cosmoem, Kartana


**Perfil de cada grupo (junto con los tipos que sobresalen y sus debilidades promedio):**

- **Titanes:** todas las estadísticas de 0.9 a 1.4 desviaciones sobre el promedio, y los extremos en rareza: pasos de huevo +2.6 (unos 25,700 pasos contra ~5,000) y captura -1.1 (promedio 18). Su felicidad base es la más baja (36 contra ~66-70 de los demás). Dominan los tipos **dragon** (x4.0 su presencia normal), **psychic** (x2.3) y **steel** (x2.0). Son los más débiles a hielo (1.38), hada (1.31) y roca (1.26), y resisten bien veneno, psíquico, planta y normal. Ojo: **no son solo legendarios**: 12 de los 71 no lo son, la mayoría pseudolegendarios (Dragonite, Tyranitar, Salamence, Metagross, Garchomp, Hydreigon, Goodra, Kommo-o); y 11 legendarios cayeron en otros grupos (Tapu Koko, Cosmog, Diancie...), porque sus estadísticas son más modestas.
- **Novatos:** todo entre -0.6 y -0.9 desviaciones, y la captura más fácil (+1.5). Ninguno es legendario. Sobresalen *bug*, *electric* y *poison*. Los más frágiles ante roca (1.24), volador (1.24) y fuego (1.20).
- **Reclutas:** ligeramente por debajo del promedio en todo (-0.2 a -0.5), sin rasgo marcado. Sobresalen *fire* y *grass*. Muy débiles a roca (1.33) y hielo (1.25).
- **Murallas:** `sp_defense` +1.1 y `defense` +0.8, pero con la velocidad más baja (-0.6). Necesitan menos experiencia para subir de nivel (-0.7 en `experience_growth`). Sobresalen *fairy* (x1.9), *steel* (x1.7) y *water* (x1.6). Resisten dragon (0.87) y normal (0.83).
- **Gladiadores:** `attack` +0.9, HP +0.7 y también defensa, y los más grandes y pesados después de los Titanes. Sobresalen *ground* (x2.4), *fighting* (x2.3) y *rock* (x2.2). Débiles a planta (1.35), lucha (1.34) y hielo (1.30), y resisten eléctrico (0.88).
- **Relámpagos:** `speed` +1.3 y `sp_attack` +0.7, con defensa promedio. Sobresalen *fire*, *flying* (x1.7 ambos) y *dark* (x1.5). Su gran debilidad es la roca (1.50) y resisten planta (0.82).

La tabla de ejemplos confirma la lectura: Relámpagos incluye a Greninja, Charizard, Sceptile y Blaziken; Gladiadores a Slaking, Gyarados, Swampert y Aggron; Murallas a Blastoise, Venusaur, Ampharos y Magnezone.

---
## 10. ¿Cuál es el mejor y el peor grupo?

"Mejor" puede significar varias cosas. El `base_total` (la suma de las estadísticas) es la medida más simple, pero no ve los **tipos**: un Pokémon con muchas estadísticas puede perder contra uno de menos si el tipo le es desfavorable. Por eso armamos una vara más completa: un **modelo de duelos 1 contra 1** entre todos los Pokémon. Para cada par (A, B) calculamos cuántos golpes necesita cada uno para tumbar al otro, y gana el que tumba en menos golpes (si empatan, el más rápido).

### 10.1 El modelo de un golpe

Usamos la fórmula de daño de los juegos con supuestos fijos: **nivel 50**, ataque de **poder 80**, bonificación **STAB** (x1.5, el Pokémon usa un ataque de su propio tipo) con el mejor de sus dos tipos, y elige entre ataque físico o especial el que más le convenga. Las estadísticas se pasan a nivel 50 con una aproximación (`base + 20` para stats y `base + 75` para HP).

Probamos con un duelo clásico a mano:

In [33]:
# 10.1 Un modelo sencillo de duelo 1 vs 1
# Para comparar grupos necesitamos una vara que mezcle TODO (stats + tipos). Usamos la fórmula de daño
# de los juegos con supuestos fijos: nivel 50, ataque de poder 80, bonificación STAB (x1.5) con el mejor
# de sus tipos, sin objetos, habilidades ni movimientos reales.
TIPO_A_COLUMNA = {t: "against_" + ("fight" if t == "fighting" else t) for t in df["type1"].unique()}


def stat_nivel50(base):
    """Aproximación de una stat (no HP) a nivel 50 con IV 31, EV 0 y naturaleza neutra."""
    return base + 20


def hp_nivel50(base):
    """Lo mismo para HP."""
    return base + 75


por_nombre = df.set_index("name", drop=False)    # para buscar Pokémon por su nombre


def dano_golpe(atacante, defensor):
    """Daño de `atacante` sobre `defensor` (por golpe), con todos los pasos a la vista."""
    a, d = por_nombre.loc[atacante], por_nombre.loc[defensor]
    tipos_atq = [a["type1"]] + ([a["type2"]] if isinstance(a["type2"], str) else [])
    eficacia = max(d[TIPO_A_COLUMNA[t]] for t in tipos_atq)         # mejor tipo propio contra el rival
    fis = 35.2 * stat_nivel50(a["attack"]) / stat_nivel50(d["defense"]) + 2
    esp = 35.2 * stat_nivel50(a["sp_attack"]) / stat_nivel50(d["sp_defense"]) + 2
    dano = max(fis, esp) * 1.5 * eficacia                           # elige el mejor entre físico y especial
    return dano, eficacia, hp_nivel50(d["hp"])


# Ejemplo a mano con dos Pokémon conocidos
for atq, rival in [("Charizard", "Blastoise"), ("Blastoise", "Charizard")]:
    dano, ef, hp_def = dano_golpe(atq, rival)
    print(f"{atq:>10} -> {rival:<10} | eficacia x{ef} | daño {dano:6.1f} | HP rival {hp_def:.0f} | golpes para KO: {int(np.ceil(hp_def / dano))}")

 Charizard -> Blastoise  | eficacia x1.0 | daño   73.0 | HP rival 154 | golpes para KO: 3
 Blastoise -> Charizard  | eficacia x2.0 | daño  138.5 | HP rival 153 | golpes para KO: 2


**En palabras simples:** Charizard (fuego/volador) no puede aprovechar su tipo contra Blastoise (agua), porque el fuego pega a la mitad y el volador pega normal (x1.0), y le necesita 3 golpes. En cambio, Blastoise le pega con agua a un Charizard de tipo fuego, que le hace el doble (x2.0), y le basta con 2. Gana Blastoise. Esa es la información que el `base_total` no ve.

### 10.2 Todos contra todos

Hacemos lo mismo, pero vectorizado, para los 801 x 801 duelos. Guardamos el resultado en una matriz `W` (1 = gana, 0 = pierde, 0.5 = empate) y, para cada Pokémon, calculamos el porcentaje de los 801 duelos que gana (`winrate`).

In [34]:
# 10.2 La misma idea, pero para TODOS contra TODOS (matriz de 801 x 801)
n = len(df)
A = df[AGAINST].values                       # multiplicador de daño que recibe cada Pokémon por tipo
col_de = {c: i for i, c in enumerate(AGAINST)}

# eficacia[i, j] = mejor multiplicador que logra el Pokémon i (con sus tipos) sobre el Pokémon j
eficacia = np.zeros((n, n))
for i, (t1, t2) in enumerate(zip(df["type1"], df["type2"])):
    tipos_i = [t1] + ([t2] if isinstance(t2, str) else [])
    eficacia[i] = np.max([A[:, col_de[TIPO_A_COLUMNA[t]]] for t in tipos_i], axis=0)

atk, dfn = stat_nivel50(df["attack"].values), stat_nivel50(df["defense"].values)
spa, spd = stat_nivel50(df["sp_attack"].values), stat_nivel50(df["sp_defense"].values)
hp50, vel = hp_nivel50(df["hp"].values), df["speed"].values

fisico = 35.2 * atk[:, None] / dfn[None, :] + 2
especial = 35.2 * spa[:, None] / spd[None, :] + 2
dano = np.maximum(fisico, especial) * 1.5 * eficacia           # dano[i, j]: i golpea a j

golpes_ko = np.ceil(hp50[None, :] / np.maximum(dano, 1e-9))    # golpes que necesita i para tumbar a j
golpes_ko[dano <= 0] = 999                                     # si no le hace daño, nunca lo tumba

# i le gana a j si lo tumba en menos golpes; si empatan, gana el más rápido
G_i, G_j = golpes_ko, golpes_ko.T
W = ((G_i < G_j) | ((G_i == G_j) & (vel[:, None] > vel[None, :]))).astype(float)
W = np.where((W + W.T) == 0, 0.5, W)                           # empate perfecto en todo -> 0.5 a cada uno
np.fill_diagonal(W, 0.5)

assert np.allclose(W + W.T, 1.0), "W debe ser antisimétrica: si uno gana, el otro pierde"
df["winrate"] = W.mean(axis=1)                                 # % de duelos ganados contra los 801

print("Matriz de duelos:", W.shape)
print("Los 10 Pokémon con más duelos ganados:")
display(df.nlargest(10, "winrate")[["name", "type1", "type2", "base_total", "winrate", "is_legendary", "grupo"]].round(3))

Matriz de duelos: (801, 801)
Los 10 Pokémon con más duelos ganados:


,name,type1,type2,base_total,winrate,is_legendary,grupo
482,Dialga,steel,dragon,680,0.963,1,Titanes
483,Palkia,water,dragon,680,0.961,1,Titanes
642,Reshiram,dragon,fire,680,0.952,1,Titanes
383,Rayquaza,dragon,flying,780,0.951,1,Titanes
486,Giratina,ghost,dragon,680,0.950,1,Titanes
444,Garchomp,dragon,ground,700,0.941,0,Titanes
717,Zygarde,dragon,ground,708,0.941,1,Titanes
791,Lunala,psychic,ghost,680,0.937,1,Titanes
716,Yveltal,dark,flying,680,0.936,1,Titanes
643,Zekrom,dragon,electric,680,0.935,1,Titanes


**Ojo con lo que NO considera el modelo:** movimientos reales, habilidades, objetos, cambios de Pokémon ni aleatoriedad. Es una vara para **comparar**, no un simulador de combates. Aun así, el resultado tiene sentido: los 10 de arriba tienen `base_total` de 680 a 780 y 8 de ellos tienen tipo *dragon* (Dialga, Palkia, Rayquaza, Garchomp...). Y la correlación entre `base_total` y `winrate` es de 0.95, o sea que el modelo concuerda en general con la suma de estadísticas pero le agrega el efecto de los tipos.

### 10.3 Mejor y peor grupo

In [35]:
# 10.3 ¿Cuál es el mejor y el peor grupo?
comparativo = df.groupby("grupo").agg(
    n=("name", "size"),
    base_total=("base_total", "mean"),
    winrate_promedio=("winrate", "mean"),
    winrate_mediana=("winrate", "median"),
).loc[ORDEN]

# Mismo cálculo pero sin legendarios, para ver qué pasa con los Pokémon "normales"
comparativo["winrate_sin_legendarios"] = df[df["is_legendary"] == 0].groupby("grupo")["winrate"].mean()
display(comparativo.round(3))

fig = px.box(df, x="grupo", y="winrate", color="grupo", category_orders={"grupo": ORDEN}, color_discrete_map=COLORES,
             points="all", hover_name="name", template=PLANTILLA, height=520,
             title="Duelos ganados (contra los 801 Pokémon) por grupo")
fig.update_layout(showlegend=False)
fig.show()

,n,base_total,winrate_promedio,winrate_mediana,winrate_sin_legendarios
grupo,,,,,
Novatos,198,289.167,0.232,0.223,0.232
Reclutas,161,363.497,0.384,0.381,0.383
Murallas,109,480.899,0.603,0.620,0.604
Gladiadores,129,492.256,0.634,0.637,0.632
Relámpagos,133,503.316,0.652,0.654,0.645
Titanes,71,626.648,0.823,0.837,0.806


In [36]:
mejor = comparativo["winrate_promedio"].idxmax()
peor = comparativo["winrate_promedio"].idxmin()
mejor_sin_leg = comparativo["winrate_sin_legendarios"].idxmax()
print(f"Mejor grupo en general : {mejor} (gana {comparativo.loc[mejor, 'winrate_promedio'] * 100:.1f}% de sus duelos)")
print(f"Peor grupo en general  : {peor} (gana {comparativo.loc[peor, 'winrate_promedio'] * 100:.1f}% de sus duelos)")
print(f"Mejor grupo SIN legendarios: {mejor_sin_leg} (gana {comparativo.loc[mejor_sin_leg, 'winrate_sin_legendarios'] * 100:.1f}%)")

# ¿Ambos criterios (base_total y duelos) ordenan igual a los grupos?
print("\nOrden por base_total  :", list(comparativo["base_total"].sort_values(ascending=False).index))
print("Orden por duelos      :", list(comparativo["winrate_promedio"].sort_values(ascending=False).index))

Mejor grupo en general : Titanes (gana 82.3% de sus duelos)
Peor grupo en general  : Novatos (gana 23.2% de sus duelos)
Mejor grupo SIN legendarios: Titanes (gana 80.6%)

Orden por base_total  : ['Titanes', 'Relámpagos', 'Gladiadores', 'Murallas', 'Reclutas', 'Novatos']
Orden por duelos      : ['Titanes', 'Relámpagos', 'Gladiadores', 'Murallas', 'Reclutas', 'Novatos']


**Respuestas:**

- **Mejor grupo: Titanes.** Ganan en promedio el **82.3%** de sus duelos y, aun quitando a los legendarios, siguen en **80.6%** (los pseudolegendarios como Garchomp o Metagross los mantienen arriba). Además es el grupo sin punto débil: su peor integrante (Wailord) gana el 59% de sus duelos, casi lo mismo que el promedio de Murallas (60%) y mucho más que la mediana de Reclutas o Novatos.
- **Peor grupo: Novatos.** Ganan solo el **23.2%** de sus duelos, y su mediana (22.3%) es casi igual al promedio, así que no es un efecto de unos cuantos casos extremos.

**Matices importantes:**

- El orden por duelos coincide con el orden por `base_total`: Titanes > Relámpagos > Gladiadores > Murallas > Reclutas > Novatos.
- Entre los tres grupos de la mitad (Relámpagos 65%, Gladiadores 63%, Murallas 60%) las diferencias son de 2 a 5 puntos y, como vimos en 6.3, Murallas y Gladiadores se traslapan con sus vecinos. Ahí **no conviene declarar un ganador claro**; son tres estilos distintos con un rendimiento parecido.
- Dentro de cada grupo hay mucha dispersión: hay Novatos que ganan el 75% de sus duelos (Togedemaru) y Murallas que ganan el 9% (Chansey). El grupo describe una tendencia, no a cada Pokémon.

---
## 11. El equipo de ensueño

La pregunta: ¿se puede armar un equipo de 6 que **inflija el mayor daño** sin dejar de ser **relativamente impermeable** a cualquier otro equipo de 6?

**Cómo lo traducimos a números** (usando la matriz de duelos `W` y las columnas `against_*`):

- **Daño:** `winrate` del equipo = promedio de duelos ganados por sus 6 integrantes contra los 801 Pokémon.
- **Impermeabilidad (cobertura):** la fracción de los 801 rivales a los que le gana **al menos uno** de nuestros 6. Si es 100%, no hay un solo Pokémon al que el equipo no tenga respuesta.
- **Debilidades compartidas:** para cada uno de los 18 tipos de ataque, contamos cuántos integrantes son débiles menos cuántos resisten. Un equipo con 4 Pokémon débiles a *ice* es fácil de explotar, aunque cada uno sea muy fuerte. Esto se penaliza.

**Puntaje del equipo = cobertura + winrate - penalización.** Es una definición nuestra (hay otras posibles), pero deja claras las dos cosas que pide el enunciado.

In [37]:
# 11.1 ¿Cómo medimos un "equipo de ensueño"?
# Queremos dos cosas a la vez:
#   (1) que dé mucho daño        -> el equipo gana muchos duelos (win rate promedio de sus 6 miembros)
#   (2) que sea "impermeable"    -> para CADA rival posible, al menos un miembro del equipo le gana (cobertura)
#                                   y que el equipo no comparta debilidades de tipo (penalización)
def cobertura(equipo, minimo=1):
    """Fracción de los 801 rivales a los que le ganan al menos `minimo` miembros del equipo."""
    return ((W[equipo] > 0.5).sum(axis=0) >= minimo).mean()


def winrate_equipo(equipo):
    """Promedio de duelos ganados por los miembros del equipo contra todos los Pokémon."""
    return W[equipo].mean()


def penalizacion_debilidades(equipo):
    """Para cada tipo de ataque: cuántos miembros son débiles menos cuántos resisten (solo si es positivo)."""
    debiles = (A[equipo] > 1).sum(axis=0)
    resisten = (A[equipo] < 1).sum(axis=0)
    return np.maximum(debiles - resisten, 0).sum() / (18 * 6)


def puntaje(equipo):
    return cobertura(equipo) + winrate_equipo(equipo) - penalizacion_debilidades(equipo)


def busca_equipo(candidatos, n_reinicios=40, semilla=RANDOM_STATE):
    """Búsqueda local: empieza con 6 al azar y cambia un miembro a la vez mientras el puntaje mejore."""
    rng = np.random.default_rng(semilla)
    mejor_eq, mejor_p = None, -np.inf
    for _ in range(n_reinicios):
        equipo = list(rng.choice(candidatos, 6, replace=False))
        p = puntaje(equipo)
        hubo_mejora = True
        while hubo_mejora:
            hubo_mejora = False
            for pos in range(6):
                for cand in candidatos:
                    if cand in equipo:
                        continue
                    prueba = equipo.copy()
                    prueba[pos] = cand
                    p_prueba = puntaje(prueba)
                    if p_prueba > p + 1e-12:
                        equipo, p, hubo_mejora = prueba, p_prueba, True
        if p > mejor_p:
            mejor_eq, mejor_p = equipo, p
    return mejor_eq, mejor_p

### 11.1 Cómo buscamos el equipo

Hay más de 10^14 equipos posibles de 6 entre 801 Pokémon, no se pueden probar todos. Usamos una **búsqueda local con reinicios**: arrancamos con 6 al azar, y vamos cambiando un integrante a la vez por el que más suba el puntaje, hasta que ningún cambio mejore. Repetimos 40 veces desde equipos distintos y nos quedamos con el mejor.

Lo corremos dos veces: con y sin legendarios (en los juegos competitivos casi siempre están prohibidos y el enunciado no dice cuál se prefiere).

In [38]:
idx_sin_leg = df.index[df["is_legendary"] == 0].values
idx_todos = df.index.values

EQUIPO_SIN_LEG, p1 = busca_equipo(idx_sin_leg)
EQUIPO_CON_LEG, p2 = busca_equipo(idx_todos)

COLS_EQ = ["name", "type1", "type2", "hp", "attack", "sp_attack", "speed", "grupo", "winrate"]
print(f"EQUIPO DE ENSUEÑO SIN LEGENDARIOS (puntaje {p1:.3f})")
display(df.loc[EQUIPO_SIN_LEG, COLS_EQ].round(3))
print(f"EQUIPO DE ENSUEÑO CON LEGENDARIOS (puntaje {p2:.3f})")
display(df.loc[EQUIPO_CON_LEG, COLS_EQ].round(3))

EQUIPO DE ENSUEÑO SIN LEGENDARIOS (puntaje 1.902)


,name,type1,type2,hp,attack,sp_attack,speed,grupo,winrate
129,Gyarados,water,flying,95,155,70,81,Gladiadores,0.931
372,Salamence,dragon,flying,95,145,120,120,Titanes,0.916
375,Metagross,steel,psychic,80,145,105,110,Titanes,0.932
288,Slaking,normal,NaN,150,160,95,100,Gladiadores,0.880
444,Garchomp,dragon,ground,108,170,120,92,Titanes,0.941
447,Lucario,fighting,steel,70,145,140,112,Relámpagos,0.871


EQUIPO DE ENSUEÑO CON LEGENDARIOS (puntaje 1.929)


,name,type1,type2,hp,attack,sp_attack,speed,grupo,winrate
129,Gyarados,water,flying,95,155,70,81,Gladiadores,0.931
482,Dialga,steel,dragon,100,120,150,90,Titanes,0.963
383,Rayquaza,dragon,flying,105,180,180,115,Titanes,0.951
375,Metagross,steel,psychic,80,145,105,110,Titanes,0.932
492,Arceus,normal,NaN,120,120,120,120,Titanes,0.906
715,Xerneas,fairy,NaN,126,131,131,99,Titanes,0.891


In [39]:
# 11.1b ¿El equipo depende de la semilla? Repetimos la búsqueda con otras semillas
for semilla in [1, 2, 3]:
    eq, p = busca_equipo(idx_sin_leg, n_reinicios=20, semilla=semilla)
    print(f"semilla {semilla}: puntaje {p:.3f} | {sorted(df.loc[eq, 'name'])}")
print("semilla base:", f"puntaje {p1:.3f} |", sorted(df.loc[EQUIPO_SIN_LEG, "name"]))

semilla 1: puntaje 1.902 | ['Garchomp', 'Gyarados', 'Lucario', 'Metagross', 'Salamence', 'Slaking']
semilla 2: puntaje 1.902 | ['Garchomp', 'Gyarados', 'Lucario', 'Metagross', 'Salamence', 'Slaking']
semilla 3: puntaje 1.902 | ['Garchomp', 'Gyarados', 'Lucario', 'Metagross', 'Salamence', 'Slaking']
semilla base: puntaje 1.902 | ['Garchomp', 'Gyarados', 'Lucario', 'Metagross', 'Salamence', 'Slaking']


**Cómo leer los resultados:**

- Con tres semillas distintas la búsqueda **encuentra exactamente el mismo equipo sin legendarios**, lo que indica que es una solución sólida y no un golpe de suerte de la búsqueda.
- **Sin legendarios:** Gyarados, Salamence, Metagross, Slaking, Garchomp y Lucario.
- **Con legendarios:** Gyarados, Dialga, Rayquaza, Metagross, Arceus y Xerneas.

**Ojo con Slaking:** su habilidad (*Truant*) lo obliga a descansar un turno de cada dos, y el modelo de duelos no considera habilidades. En un combate real no sería una buena elección. Lo dejamos porque es lo que sale con las reglas que definimos, pero hay que decirlo.

In [40]:
# 11.2 Equipos de referencia para comparar
TOP6_BASE_TOTAL = list(df.nlargest(6, "base_total").index)               # los 6 con mayor suma de stats
TOP6_WINRATE = list(df.nlargest(6, "winrate").index)                     # los 6 que más duelos ganan
TOP6_WINRATE_SIN_LEG = list(df[df["is_legendary"] == 0].nlargest(6, "winrate").index)   # lo mismo, sin legendarios
rng = np.random.default_rng(RANDOM_STATE)
ALEATORIO = list(rng.choice(idx_todos, 6, replace=False))                # 6 al azar

equipos = {
    "Ensueño (sin legendarios)": EQUIPO_SIN_LEG,
    "Ensueño (con legendarios)": EQUIPO_CON_LEG,
    "Top 6 por base_total": TOP6_BASE_TOTAL,
    "Top 6 por duelos ganados": TOP6_WINRATE,
    "Top 6 por duelos (sin legendarios)": TOP6_WINRATE_SIN_LEG,
    "6 al azar": ALEATORIO,
}

filas = []
for nombre, eq in equipos.items():
    filas.append({
        "equipo": nombre,
        "winrate": winrate_equipo(eq),
        "cobertura (>=1 le gana)": cobertura(eq),
        "cobertura robusta (>=3)": cobertura(eq, minimo=3),
        "penalización debilidades": penalizacion_debilidades(eq),
        "tipo con más débiles": int((A[eq] > 1).sum(axis=0).max()),
        "puntaje": puntaje(eq),
    })
tabla_equipos = pd.DataFrame(filas).set_index("equipo")
display(tabla_equipos.round(3))

print("Top 6 por base_total :", [df.loc[i, "name"] for i in TOP6_BASE_TOTAL])
print("Top 6 por duelos     :", [df.loc[i, "name"] for i in TOP6_WINRATE])
print("Top 6 por duelos (sin legendarios):", [df.loc[i, "name"] for i in TOP6_WINRATE_SIN_LEG])

,winrate,cobertura (>=1 le gana),cobertura robusta (>=3),penalización debilidades,tipo con más débiles,puntaje
equipo,,,,,,
Ensueño (sin legendarios),0.912,1.000,0.975,0.009,2,1.902
Ensueño (con legendarios),0.929,1.000,0.989,0.000,2,1.929
Top 6 por base_total,0.893,1.000,0.981,0.074,3,1.819
Top 6 por duelos ganados,0.953,0.999,0.988,0.139,5,1.813
Top 6 por duelos (sin legendarios),0.923,1.000,0.978,0.074,3,1.849
6 al azar,0.443,0.823,0.497,0.056,3,1.210


Top 6 por base_total : ['Mewtwo', 'Rayquaza', 'Kyogre', 'Groudon', 'Arceus', 'Zygarde']
Top 6 por duelos     : ['Dialga', 'Palkia', 'Reshiram', 'Rayquaza', 'Giratina', 'Garchomp']
Top 6 por duelos (sin legendarios): ['Garchomp', 'Metagross', 'Gyarados', 'Gengar', 'Salamence', 'Gallade']


In [41]:
# 11.3 Prueba final: equipo contra equipo
# Un equipo A le gana a un equipo B si gana más de la mitad de los 36 duelos posibles (6 x 6).
def duelo_equipos(eq, rivales):
    """Fracción de los 36 duelos que gana `eq` contra cada equipo rival (rivales: matriz [n_equipos, 6])."""
    return W[eq][:, rivales].mean(axis=(0, 2))


N_RIVALES = 5000
rng = np.random.default_rng(RANDOM_STATE)
rivales_azar = np.array([rng.choice(idx_todos, 6, replace=False) for _ in range(N_RIVALES)])

# Rivales "difíciles": equipos armados al azar pero SOLO con los 100 Pokémon que más duelos ganan
top100 = df.nlargest(100, "winrate").index.values
rivales_elite = np.array([rng.choice(top100, 6, replace=False) for _ in range(N_RIVALES)])

filas = []
for nombre, eq in equipos.items():
    for tipo_rival, rivales in [("al azar", rivales_azar), ("de élite (top 100)", rivales_elite)]:
        frac = duelo_equipos(eq, rivales)
        filas.append({"equipo": nombre, "rivales": tipo_rival,
                      "% de duelos ganados": frac.mean() * 100,
                      "% de equipos rivales vencidos": (frac > 0.5).mean() * 100,
                      "peor resultado (% duelos)": frac.min() * 100})
resultado_vs = pd.DataFrame(filas)
display(resultado_vs.round(1))

fig = px.bar(resultado_vs, x="equipo", y="% de equipos rivales vencidos", color="rivales", barmode="group",
             template=PLANTILLA, height=480, title="% de equipos rivales a los que cada equipo logra vencer")
fig.show()

,equipo,rivales,% de duelos ganados,% de equipos rivales vencidos,peor resultado (% duelos)
0,Ensueño (sin legendarios),al azar,91.2,100.0,59.7
1,Ensueño (sin legendarios),de élite (top 100),64.3,93.5,36.1
2,Ensueño (con legendarios),al azar,92.9,100.0,65.3
3,Ensueño (con legendarios),de élite (top 100),72.6,99.4,41.7
4,Top 6 por base_total,al azar,89.3,100.0,58.3
5,Top 6 por base_total,de élite (top 100),68.7,98.0,36.1
6,Top 6 por duelos ganados,al azar,95.4,100.0,58.3
7,Top 6 por duelos ganados,de élite (top 100),77.8,99.5,43.1
8,Top 6 por duelos (sin legendarios),al azar,92.3,100.0,56.9
9,Top 6 por duelos (sin legendarios),de élite (top 100),67.6,95.9,33.3


In [42]:
# 11.4 Los dos equipos de ensueño frente a frente
print("Sin legendarios vs con legendarios: gana (duelos de 36) ->",
      f"{W[np.ix_(EQUIPO_SIN_LEG, EQUIPO_CON_LEG)].mean() * 100:.1f}% el equipo sin legendarios")

# Perfil defensivo del equipo: daño recibido por cada tipo de ataque
nombres_eq = df.loc[EQUIPO_SIN_LEG, "name"].values
tabla_def = pd.DataFrame(A[EQUIPO_SIN_LEG], index=nombres_eq, columns=[c.replace("against_", "") for c in AGAINST])

fig = px.imshow(tabla_def, text_auto=".2g", color_continuous_scale="RdBu_r", color_continuous_midpoint=1, aspect="auto",
                title="Daño que recibe cada miembro del equipo (sin legendarios) por tipo de ataque",
                template=PLANTILLA, height=420)
fig.show()

debiles = (A[EQUIPO_SIN_LEG] > 1).sum(axis=0)
resisten = (A[EQUIPO_SIN_LEG] < 1).sum(axis=0)
resumen_def = pd.DataFrame({"débiles": debiles, "resisten": resisten}, index=tabla_def.columns).T
display(resumen_def)

Sin legendarios vs con legendarios: gana (duelos de 36) -> 40.3% el equipo sin legendarios


,bug,dark,dragon,electric,fairy,fight,fire,flying,ghost,grass,ground,ice,normal,poison,psychic,rock,steel,water
débiles,0,1,2,1,2,2,2,0,1,0,2,2,0,0,0,2,0,0
resisten,3,1,2,1,1,2,3,1,1,3,2,2,2,3,1,3,3,2


**Cómo leer las tablas y las gráficas:**

**Contra equipos al azar:** el equipo sin legendarios gana el **91%** de los duelos y vence al 100% de los 5,000 equipos rivales; incluso en su peor caso gana el 60% de los duelos. Un equipo de 6 al azar apenas gana el 44%.

**Contra equipos "de élite"** (armados con los 100 Pokémon que más duelos ganan, es decir, rivales difíciles): gana el 64% de los duelos y vence al **93.5%** de los equipos rivales, pero su peor caso baja a 36%. **No es invencible.** Y contra el equipo de ensueño *con* legendarios gana solo el 40% de los duelos.

**Comparación con los equipos de referencia (todos sin legendarios, para ser justos):** el "top 6 por duelos ganados" sin legendarios (Garchomp, Metagross, Gyarados, Gengar, Salamence, Gallade) pega algo más fuerte (92.3% vs 91.2% contra equipos al azar, 67.6% vs 64.3% contra los de élite), pero tiene debilidades encimadas: hasta **3** integrantes débiles al mismo tipo, y una penalización de 0.074 contra 0.009 del equipo de ensueño. En el peor caso contra equipos de élite también pierde más (33% de duelos ganados vs 36%).

**Defensa del equipo sin legendarios (última gráfica):** nunca hay más de **2** integrantes débiles al mismo tipo, y para casi todos los tipos hay al menos tantos integrantes que resisten como débiles. La única excepción es *fairy* (2 débiles y 1 que resiste).

**Respuesta a la pregunta:** **sí, es posible**, con matices. Se puede armar un equipo que a) gana la gran mayoría de los duelos, b) tiene respuesta para **todos** los Pokémon del dataset (cobertura del 100%, y el 97.5% de los rivales tienen al menos 3 Pokémon del equipo que les ganan) y c) no tiene debilidades encimadas. Pero "relativamente impermeable" no es "invencible": frente a equipos de élite pierde alrededor de 1 de cada 3 duelos, y el equipo está optimizado contra los **Pokémon individuales**, no contra combinaciones de 6 pensadas para vencerlo. Además, depende de cómo definimos el puntaje: si priorizáramos solo el daño, saldría un equipo distinto (y con más debilidades).

---
## 12. Conclusiones y limitaciones

**Resultados:**

- Dividimos a los 801 Pokémon en **6 grupos** (K-Means sobre 11 variables escaladas): **Novatos, Reclutas, Murallas, Gladiadores, Relámpagos y Titanes**.
- **Mejor grupo: Titanes** (gana el 82% de los duelos). **Peor grupo: Novatos** (gana el 23%).
- **Equipo de ensueño:** sí es posible armar uno con cobertura total y sin debilidades encimadas. Sin legendarios: Gyarados, Salamence, Metagross, Slaking, Garchomp y Lucario.
- Los grupos reflejan sobre todo un **gradiente de poder** (PC1 explica el 43% de la varianza), sobre el que se superponen estilos de combate: velocidad, defensa, fuerza física.

**Limitaciones a tener en cuenta:**

- *Los grupos no son "naturales":* el silhouette global es de 0.145 y los grupos del "medio alto" (sobre todo Murallas y Gladiadores) se traslapan. Son un resumen útil, no fronteras duras.
- *El número de grupos es una decisión:* los índices de calidad prefieren *k* = 2 o 3, que solo separan por poder; escogimos *k* = 6 para poder perfilar estilos.
- *Las variables que entran son una decisión:* con solo estadísticas o con otros conjuntos, los grupos cambian (sección 4.4).
- *El modelo de duelos es una simplificación:* no considera movimientos, habilidades, objetos ni cambios, y supone nivel 50 y un ataque de poder fijo. Por eso, por ejemplo, Slaking sale en el equipo.
- *El equipo de ensueño depende de nuestro puntaje* (cobertura + winrate - penalización) y se evaluó contra rivales individuales y equipos armados al azar; no contra estrategias reales.
- *Datos:* faltan 20 alturas y pesos (imputados con la mediana), y el archivo trae 801 Pokémon y no 802.

In [43]:
# Guardamos la tabla final con el grupo de cada Pokémon (útil para armar la presentación)
SALIDA = "pokemon_grupos.csv"
df[["pokedex_number", "name", "type1", "type2", "grupo", "base_total", "winrate", "pc1", "pc2", "u1", "u2"]].to_csv(SALIDA, index=False)
print("Guardado en", SALIDA)

Guardado en pokemon_grupos.csv
